# 🔬 OW-OVD Pest Detection Demo on Kaggle
Notebook này hướng dẫn bạn cách thiết lập môi trường, chuẩn bị đặc trưng embeddings và khởi chạy ứng dụng web demo trực tiếp trên Kaggle GPU.

### ⚠️ Yêu cầu trước khi chạy:
1. Hãy chắc chắn rằng bạn đã kích hoạt **GPU T4 x2** hoặc **GPU P100** trong phần settings của Kaggle (bên phải màn hình: *Accelerator -> GPU*).
2. Bật kết nối internet cho notebook (*Internet on*).

## Bước 1: Clone Repository từ GitHub
Thay đổi biến `repo_url` dưới đây thành link GitHub chứa code của bạn nếu cần.

In [ ]:
# Khai báo thông tin repo
import os
repo_url = "https://github.com/nta2112/OW_OVD-An-custom.git"
working_dir = "/kaggle/working/OW_OVD"

if not os.path.exists(working_dir):
    print("-> Đang clone repository từ GitHub...")
    !git clone {repo_url} {working_dir}
else:
    print("-> Repository đã tồn tại. Đang tiến hành cập nhật (git pull)...")
    %cd {working_dir}
    !git pull

%cd {working_dir}

# Tải mmyolo vào thư mục third_party nếu chưa có
if not os.path.exists("third_party/mmyolo"):
    print("-> Đang tải submodule mmyolo...")
    !git clone https://github.com/open-mmlab/mmyolo.git third_party/mmyolo
else:
    print("-> Submodule mmyolo đã có sẵn.")

## Bước 2: Cài đặt Môi trường & Thư viện (MMCV, MMDetection, Gradio)
Cell này tự động hạ cấp PyTorch về bản ổn định `2.4.0+cu121` và cài đặt các phiên bản build sẵn tương thích của MMCV để tránh việc phải build từ source mất 20 phút.

In [ ]:
print("-> 1. Thiết lập phiên bản PyTorch & Torchvision...")
!pip install torch==2.4.0+cu121 torchvision==0.19.0+cu121 --extra-index-url https://download.pytorch.org/whl/cu121

print("\n-> 2. Cài đặt MMCV từ wheel index...")
!pip install mmcv -f https://download.openmmlab.com/mmcv/dist/cu121/torch2.4/index.html

print("\n-> 3. Cài đặt các thư viện bổ trợ & Gradio...")
!pip install matplotlib pycocotools terminaltables mmengine prettytable wcwidth open_clip_torch "transformers<4.41.0" "gradio>=4.0.0"

print("\n-> 4. Cài đặt MMDetection...")
!pip install "mmdet>=3.1.0" --no-deps

print("\n-> 5. Cài đặt MMYOLO từ source...")
!pip install --no-build-isolation --no-deps third_party/mmyolo

print("\n-> Kiểm tra import xem tất cả các package đã ổn định chưa...")
import torch, mmcv, mmdet, mmyolo, gradio
print(f"  - torch: {torch.__version__} (CUDA: {torch.cuda.is_available()})")
print(f"  - mmcv: {mmcv.__version__}")
print(f"  - mmdet: {mmdet.__version__}")
print(f"  - mmyolo: {mmyolo.__version__}")
print(f"  - gradio: {gradio.__version__}")
print("====== Cài đặt môi trường thành công! ======")

## Bước 3: Tạo thư mục dữ liệu & Sinh đặc trưng Embeddings bằng CLIP
Cell này đọc danh sách class từ file annotations của IP102, sinh ra đặc trưng văn bản bằng model CLIP và tạo các file thuộc tính + phân phối mẫu giống hệt lúc huấn luyện.

In [ ]:
import json
import torch
import numpy as np
import os
from transformers import AutoTokenizer, CLIPTextModelWithProjection

# 1. Khởi tạo các thư mục
os.makedirs('pretrained_models', exist_ok=True)
os.makedirs('data/IP102', exist_ok=True)
os.makedirs('data/texts/IP102', exist_ok=True)

# 2. Tải pretrain weights của YOLO-World
weights_path = 'pretrained_models/yolo_world_v2_l_obj365v1_goldg_pretrain-a82b1fe3.pth'
if not os.path.exists(weights_path):
    print("-> Đang tải pretrained weights...")
    !wget -O {weights_path} https://huggingface.co/wondervictor/YOLO-World/resolve/main/yolo_world_v2_l_obj365v1_goldg_pretrain-a82b1fe3.pth
else:
    print("-> Pretrained weights đã có sẵn.")

# 3. Đọc tên các class từ file annotations IP102
ann_path = '/kaggle/input/datasets/eljazouly/ip102-coco-annotations/coco_annotations/train.json'
if os.path.exists(ann_path):
    with open(ann_path, 'r') as f:
        coco_data = json.load(f)
    categories = sorted(coco_data['categories'], key=lambda x: x['id'])
    class_names = [cat['name'] for cat in categories]
else:
    print("-> Không tìm thấy file annotations trên Kaggle dataset. Sử dụng danh sách class fallback mặc định...")
    # Fallback list lấy trực tiếp từ config/demo_app
    from demo_app import IP102_CLASSES
    class_names = IP102_CLASSES[:102]

num_classes = len(class_names)
print(f"-> Tổng số lớp học (classes): {num_classes}")

# 4. Lưu class_texts.json
class_texts = [[name] for name in class_names]
with open('data/texts/IP102/class_texts.json', 'w') as f:
    json.dump(class_texts, f)

# 5. Sinh class embeddings bằng CLIP
print("-> Đang trích xuất text embeddings bằng CLIP (openai/clip-vit-base-patch32)...")
model_name = 'openai/clip-vit-base-patch32'
tokenizer = AutoTokenizer.from_pretrained(model_name)
clip_model = CLIPTextModelWithProjection.from_pretrained(model_name, use_safetensors=True)
clip_model.eval()

embeddings = []
with torch.no_grad():
    for name in class_names:
        inputs = tokenizer(name, padding=True, return_tensors="pt")
        outputs = clip_model(**inputs)
        embed = outputs.text_embeds[0].cpu().numpy()
        embed = embed / np.linalg.norm(embed)
        embeddings.append(embed)

np.save('data/IP102/ip102_gt_embeddings.npy', np.array(embeddings))
print("-> Đã lưu ip102_gt_embeddings.npy")

# 6. Sinh file task_att_1_embeddings.pth
num_att = num_classes * 25
torch.save({
    'att_embedding': torch.zeros(num_att, 512),
    'att_text': [f"att_{i}" for i in range(num_att)]
}, 'data/IP102/task_att_1_embeddings.pth')
print("-> Đã lưu task_att_1_embeddings.pth")

# 7. Sinh file mowod_distribution_sim1.pth
thrs = [0.55]
pos_dist = [{att_i: torch.zeros(10000) for att_i in range(num_att)} for _ in thrs]
neg_dist = [{att_i: torch.zeros(10000) for att_i in range(num_att)} for _ in thrs]
torch.save({
    'positive_distributions': pos_dist,
    'negative_distributions': neg_dist
}, 'data/IP102/mowod_distribution_sim1.pth')
print("-> Đã sinh tất cả các file embeddings thành công!")

## Bước 4: Khởi chạy Ứng dụng Demo Gradio
Chỉ định đường dẫn checkpoint bạn đã huấn luyện (ví dụ: `t3_best.pth`). Lệnh này sử dụng flag `--share` để sinh ra link công khai dạng `https://xxxx.gradio.live` giúp bạn click truy cập từ bất cứ đâu.

In [ ]:
# Đường dẫn tới checkpoint tốt nhất của bạn trên Kaggle
# Ví dụ đường dẫn checkpoint nếu bạn add dataset vào: 
# "/kaggle/input/.../t3_best.pth" hoặc một checkpoint local trong working directory
checkpoint_path = "/kaggle/working/OW_OVD/pretrained_models/yolo_world_v2_l_obj365v1_goldg_pretrain-a82b1fe3.pth"  # Thay bằng path checkpoint của bạn

config_path = "configs/open_world/mowod/custom/ip102_t3.py"
ann_file_path = "/kaggle/input/datasets/eljazouly/ip102-coco-annotations/coco_annotations/train.json"

# Khởi chạy Gradio app
if not os.path.exists(checkpoint_path):
    print(f"⚠️ Cảnh báo: Không tìm thấy checkpoint tại {checkpoint_path}.")
    print("Hệ thống sẽ chạy thử ở chế độ UI-only (--no-model) để test giao diện web.")
    !python demo_app.py --config {config_path} --checkpoint {checkpoint_path} --no-model --share
else:
    # Dùng annotation file nếu tồn tại trên Kaggle để lấy tên lớp chuẩn xác
    ann_arg = f"--ann-file {ann_file_path}" if os.path.exists(ann_file_path) else ""
    !python demo_app.py --config {config_path} --checkpoint {checkpoint_path} {ann_arg} --device cuda:0 --share